In [5]:
# %%
"""
QAT — Real INT8 Inference via onednn (CPU)
===========================================
Trains with QAT (fake-quant on GPU + AMP), then converts to
REAL INT8 using onednn and evaluates on CPU.

Sweep A : Observer type  (minmax vs moving_avg)  with lr=constant
Sweep B : LR schedule    (cosine, step)           with best observer from A
Best result : highest INT8 accuracy across all configs

FIXES carried forward from previous version:
  Fix 1 — NUM_WORKERS=0 inside Jupyter/IPython on Windows (spawn issue).
  Fix 2 — prepare_qat_fx example_inputs as tuple-of-args, not bare tensor.
  Fix 3 — GradScaler constructed with device="cuda" keyword (PyTorch 2.x API).
  Fix 4 — No _orig_mod fishing; prepared GraphModule used directly.
  Fix 5 — freeze_bn_stats replaced by manual BN.eval() (torch.ao builds).
  Fix 6 — worker_init_fn only used when NUM_WORKERS > 0.

ADDITIONAL FIXES in this version:
  Fix 7 — count_params: uses canonical two-path deduplication (id(p) for
           nn.Parameter, module-path key for FX callable weights).
           Previously used bare sum(p.numel()) on fp32_model only —
           never measured the actual INT8 model.
  Fix 8 — disk_size_mb: uses torch.save(model) not state_dict() so FP32
           and INT8 models are serialised identically for fair comparison.
  Fix 9 — count_flops: fvcore preferred, thop fallback, manual hook as
           last resort. FP32 FLOPs computed once and reused for INT8
           (quantization does not change op count).
  Fix 10 — measure_latency_cpu: warmup raised to 20 (matches CPU PTQ
            script), canonical output keys used consistently.
  Fix 11 — JSON schema deduplicated: metrics live under "int8" and
            "fp32_reference" only, no duplicate top-level copies.
  Fix 12 — FP32 reference timing measured ONCE before the sweep and
            reused, not re-timed inside every run_one() call.
  Fix 13 — _best_result section added to JSON output.
  Fix 14 — test loader batch_size aligned to BATCH_SIZE (128) for
            consistency with timing benchmarks.

Mathematical foundation:
  Fake-quantization (QAT forward pass):
    Q(x) = clamp(round(x / S) + Z, q_min, q_max)
    x̂   = S · (Q(x) − Z)
    Gradient flows through clamp via Straight-Through Estimator (STE):
    ∂L/∂x ≈ ∂L/∂x̂ · 1[q_min ≤ Q(x) ≤ q_max]

  MinMaxObserver:
    S = (x_max − x_min) / (2^8 − 1)
    Z = clip(round(−x_min / S), 0, 255)

  MovingAverageMinMaxObserver:
    x_min_t = α · x_min_{t-1} + (1−α) · min(x_t)   [EMA over batches]
    S, Z computed from smoothed x_min_t / x_max_t
    More stable than MinMax for distributions that shift during training.

  Per-channel weight quantization:
    Each output filter i has its own (Sᵢ, Zᵢ) → lower reconstruction
    error than a single global scale shared across all filters.

FLOPs note:
  QAT does not change the computation graph — GFLOPs identical to FP32.
  INT8 CPU speedup comes from onednn VNNI/SIMD packing, not fewer ops.

Output:
  __2__QAT.json
  __2__saved_models_QAT/
"""

# %%
import os
import sys
import json
import time
import copy
import tempfile
import warnings
import random

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score
)

# fvcore preferred for FLOPs; thop as fallback
try:
    from fvcore.nn import FlopCountAnalysis
    _FVCORE = True
except ImportError:
    _FVCORE = False
    try:
        from thop import profile as thop_profile
        _THOP = True
    except ImportError:
        _THOP = False

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── INT8 backend ──────────────────────────────────────────────────────────────
torch.backends.quantized.engine = "onednn"

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
cap      = torch.cuda.get_device_capability() if DEVICE.type == "cuda" else (0, 0)
SM       = f"SM{cap[0]}{cap[1]}" if DEVICE.type == "cuda" else "N/A"

print(f"[init] PyTorch : {torch.__version__}")
print(f"[init] Device  : {DEVICE}")
print(f"[init] GPU     : {GPU_NAME}  ({SM})")
print(f"[init] INT8    : onednn (CPU real INT8 SIMD kernels)")
print(f"[init] Seed    : {SEED}")

# ── QAT imports ───────────────────────────────────────────────────────────────
from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx
from torch.ao.quantization import (
    QConfig,
    MinMaxObserver,
    PerChannelMinMaxObserver,
    MovingAverageMinMaxObserver,
    MovingAveragePerChannelMinMaxObserver,
)
print("[init] QAT backend : FX-graph (torch.ao)")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__2__QAT.json"
SAVE_DIR              = "__2__saved_models_QAT"

BATCH_SIZE            = 128
NUM_CLASSES           = 10
QAT_EPOCHS            = 3
LR_BASE               = 1e-4
OBSERVER_FREEZE_EPOCH = 1
TIMING_RUNS           = 100
WARMUP_RUNS           = 20        # Fix 10: raised from 10 → 20
USE_AMP               = DEVICE.type == "cuda"
INPUT_SHAPE           = (1, 3, 32, 32)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# Fix 1: Jupyter/Windows spawn issue
def _is_jupyter() -> bool:
    return "ipykernel" in sys.modules

NUM_WORKERS = 0 if (_is_jupyter() and os.name == "nt") else 4
PIN_MEMORY  = DEVICE.type == "cuda" and NUM_WORKERS > 0

print(f"[init] AMP         : {USE_AMP}")
print(f"[init] Epochs      : {QAT_EPOCHS}  Batch: {BATCH_SIZE}  Timing runs: {TIMING_RUNS}")
print(f"[init] NUM_WORKERS : {NUM_WORKERS}")

# ── Observer / LR configs ─────────────────────────────────────────────────────
OBSERVER_QCONFIGS = {
    "minmax": QConfig(
        activation=MinMaxObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=PerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
    "moving_avg": QConfig(
        activation=MovingAverageMinMaxObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=MovingAveragePerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
}

LR_SCHEDULES = {
    "constant": lambda opt, ep: None,
    "cosine"  : lambda opt, ep: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ep),
    "step"    : lambda opt, ep: torch.optim.lr_scheduler.StepLR(
                    opt, step_size=max(1, ep // 2), gamma=0.1),
}


# ══════════════════════════════════════════════════════════════════════════════
# Model
# ══════════════════════════════════════════════════════════════════════════════
def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model


def load_baseline(path: str) -> nn.Module:
    print(f"[load] {path} ...", flush=True)
    model = build_model(NUM_CLASSES).cpu()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    print("[load] OK", flush=True)
    return model


# ══════════════════════════════════════════════════════════════════════════════
# Data
# ══════════════════════════════════════════════════════════════════════════════
def _worker_init_fn(worker_id: int) -> None:
    np.random.seed(SEED + worker_id)


def get_train_loader() -> DataLoader:
    tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(
        root="../data", train=True, download=True, transform=tf)
    return DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        worker_init_fn=_worker_init_fn if NUM_WORKERS > 0 else None,
    )


def get_test_loader() -> DataLoader:
    # Fix 14: batch_size=BATCH_SIZE (128) — consistent with timing benchmarks.
    # Original used 512 which is inconsistent and makes per-image stats misleading.
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(
        root="../data", train=False, download=True, transform=tf)
    return DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )


# ══════════════════════════════════════════════════════════════════════════════
# Parameter Counting  (Fix 7)
# Canonical two-path deduplication — handles FP32 and FX-quantized models.
# ══════════════════════════════════════════════════════════════════════════════
def count_params(model: nn.Module, model_label: str = "") -> dict:
    """
    ① nn.Parameter objects  → deduplicate by id(p).
      id() is safe: each Parameter is a distinct Python object.

    ② FX-quantized .weight() callables → deduplicate by (module_name, 'weight').
      convert_fx() ops expose weights via .weight() returning a fresh
      dequantized FloatTensor each call; id() and data_ptr() are unstable.

    Why NOT data_ptr():
      Views/slices share data_ptr() with their base, causing silent
      undercounting when used as a dedup key.
    """
    total, nonzero   = 0, 0
    seen_param_ids   = set()
    seen_module_keys = set()

    for mod_name, module in model.named_modules():
        # ① Standard parameters
        for _, p in module.named_parameters(recurse=False):
            pid = id(p)
            if pid in seen_param_ids:
                continue
            seen_param_ids.add(pid)
            total   += p.numel()
            nonzero += int((p != 0).sum().item())

        # ② FX quantized callable weights
        if hasattr(module, "weight") and callable(module.weight):
            key = (mod_name, "weight")
            if key in seen_module_keys:
                continue
            seen_module_keys.add(key)
            try:
                w = module.weight()
                if isinstance(w, torch.Tensor) and w.numel() > 0:
                    total   += w.numel()
                    nonzero += int((w != 0).sum().item())
            except Exception:
                pass

    if model_label:
        print(
            f"      count_params [{model_label}]: "
            f"total={total:,}  nonzero={nonzero:,}  "
            f"sparsity={1 - nonzero / max(total, 1):.4f}",
            flush=True,
        )
    return {"params_total": int(total), "params_nonzero": int(nonzero)}


# ══════════════════════════════════════════════════════════════════════════════
# Disk Size  (Fix 8)
# torch.save(model) for both FP32 and INT8 → apples-to-apples comparison.
# state_dict() omits quantization config, making INT8 appear smaller than it is.
# ══════════════════════════════════════════════════════════════════════════════
def disk_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)


# ══════════════════════════════════════════════════════════════════════════════
# FLOPs  (Fix 9)
# fvcore → thop → manual hook fallback.
# Computed ONCE on FP32 and reused for INT8 (op count is identical).
# ══════════════════════════════════════════════════════════════════════════════
def count_flops(model: nn.Module) -> dict:
    model = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(*INPUT_SHAPE)

    if _FVCORE:
        try:
            fa = FlopCountAnalysis(model, dummy)
            fa.unsupported_ops_warnings(False)
            macs = fa.total()
            params_total = sum(p.numel() for p in model.parameters())
            return {
                "flops_G"     : round(macs * 2 / 1e9, 9),
                "params_total": int(params_total),
                "tool"        : "fvcore",
            }
        except Exception:
            pass

    if _THOP:
        try:
            macs, _ = thop_profile(model, inputs=(dummy,), verbose=False)
            params_total = sum(p.numel() for p in model.parameters())
            return {
                "flops_G"     : round(float(macs) * 2 / 1e9, 9),
                "params_total": int(params_total),
                "tool"        : "thop",
            }
        except Exception:
            pass

    # Manual hook fallback
    spatial, handles = {}, []

    def _hook(name):
        def h(_, __, out):
            if hasattr(out, "shape"):
                spatial[name] = tuple(out.shape)
        return h

    for name, mod in model.named_modules():
        handles.append(mod.register_forward_hook(_hook(name)))
    with torch.no_grad():
        model(dummy)
    for h in handles:
        h.remove()

    total_flops = 0
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Conv2d):
            shape = spatial.get(name)
            if shape is None:
                continue
            _, cout, hout, wout = shape
            kh = mod.kernel_size[0] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            kw = mod.kernel_size[1] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            total_flops += 2 * cout * (mod.in_channels // mod.groups) * kh * kw * hout * wout
        elif isinstance(mod, nn.Linear):
            total_flops += 2 * mod.in_features * mod.out_features

    params_total = sum(p.numel() for p in model.parameters())
    return {
        "flops_G"     : round(total_flops / 1e9, 9),
        "params_total": int(params_total),
        "tool"        : "manual",
    }


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader,
             device: torch.device = torch.device("cpu")) -> dict:
    model = model.to(device).eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        out = model(inputs.to(device))
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred = np.array(preds)
    y_true = np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


# ══════════════════════════════════════════════════════════════════════════════
# CPU Timing  (Fix 10)
# Canonical schema, warmup=20, consistent keys across all scripts.
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def measure_latency_cpu(model: nn.Module, n: int = TIMING_RUNS,
                        label: str = "model") -> dict:
    model    = copy.deepcopy(model).cpu().eval()
    dummy_1  = torch.randn(1,          3, 32, 32)
    dummy_b  = torch.randn(BATCH_SIZE, 3, 32, 32)

    # Warmup — stabilise OS scheduler, branch predictor, caches
    for _ in range(WARMUP_RUNS):
        model(dummy_1)
        model(dummy_b)

    single_t, batch_t = [], []
    for _ in range(n):
        t = time.perf_counter()
        model(dummy_1)
        single_t.append((time.perf_counter() - t) * 1000)

    for _ in range(n):
        t = time.perf_counter()
        model(dummy_b)
        batch_t.append((time.perf_counter() - t) * 1000)

    bms = float(np.mean(batch_t))
    sms = float(np.mean(single_t))
    print(
        f"      [time:{label}] single={sms:.2f} ms  batch={bms:.1f} ms  "
        f"throughput={BATCH_SIZE / (bms / 1000):.0f} imgs/s",
        flush=True,
    )
    return {
        "single_img_cpu_ms"  : round(sms, 4),
        "batch128_cpu_ms"    : round(bms, 4),
        "per_img_cpu_ms"     : round(bms / BATCH_SIZE, 4),
        "throughput_imgs_sec": round(BATCH_SIZE / (bms / 1000), 1),
        "timing_method"      : "wall-clock (time.perf_counter) — CPU only",
        "warmup_runs"        : WARMUP_RUNS,
        "timing_runs"        : n,
        "label"              : label,
    }


# ══════════════════════════════════════════════════════════════════════════════
# Canonical Result Entry Builder
# Mirrors PTQ CPU script schema exactly.
# ══════════════════════════════════════════════════════════════════════════════
def build_entry(metrics: dict, params: dict, size_mb: float,
                flops_G: float, inference_ms: dict) -> dict:
    return {
        "accuracy"      : round(metrics["accuracy"],  6),
        "precision"     : round(metrics["precision"], 6),
        "recall"        : round(metrics["recall"],    6),
        "f1"            : round(metrics["f1"],        6),
        "params"        : int(params["params_total"]),
        "params_nonzero": int(params["params_nonzero"]),
        "size_mb"       : round(size_mb, 6),
        "flops_G"       : round(flops_G, 9),
        "inference_ms"  : inference_ms,
    }


# ══════════════════════════════════════════════════════════════════════════════
# INT8 Conversion
# ══════════════════════════════════════════════════════════════════════════════
def convert_to_int8(fakequant_model: nn.Module) -> nn.Module:
    """
    convert_fx() replaces every FakeQuantize node with a genuine quantized op
    backed by onednn INT8 SIMD kernels. This is REAL INT8, not simulated.
    The model must be on CPU and in eval() before conversion.
    """
    return convert_fx(copy.deepcopy(fakequant_model).cpu().eval())


# ══════════════════════════════════════════════════════════════════════════════
# Observer + BN Freezing
# ══════════════════════════════════════════════════════════════════════════════
def freeze_observers_and_bn(model: nn.Module) -> None:
    """
    Fix 5: torch.nn.intrinsic.qat.freeze_bn_stats removed in torch.ao builds.
    Disable observers via stable public API; freeze BN by switching to eval().
    Effect is identical: running mean/var stop updating, S/Z are fixed.
    """
    try:
        model.apply(torch.ao.quantization.disable_observer)
    except Exception as e:
        print(f"      [freeze] disable_observer warning: {e}", flush=True)

    bn_frozen = 0
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
            m.eval()
            bn_frozen += 1
    print(f"      [freeze] Froze {bn_frozen} BN layers", flush=True)


# ══════════════════════════════════════════════════════════════════════════════
# QAT Training Loop
# ══════════════════════════════════════════════════════════════════════════════
def train_qat(fp32_model: nn.Module, train_loader: DataLoader,
              obs_name: str, lr_schedule_name: str,
              n_epochs: int = QAT_EPOCHS):
    """
    Inserts FakeQuantize nodes via prepare_qat_fx (this IS QAT — weights stay
    FP32 but quantization error is simulated via STE during training).
    convert_fx() later replaces FakeQuantize nodes with real INT8 ops.

    Fix 2: example_inputs as tuple-of-args.
    Fix 3: GradScaler with device keyword.
    Fix 4: no _orig_mod fishing.
    """
    qconfig        = OBSERVER_QCONFIGS[obs_name]
    model          = copy.deepcopy(fp32_model).cpu().train()
    example_inputs = (torch.randn(1, 3, 32, 32),)          # Fix 2

    prepared = prepare_qat_fx(model, {"": qconfig}, example_inputs)
    prepared = prepared.to(DEVICE)

    # Warm-up pass: initialise observer statistics before first training step
    with torch.no_grad():
        prepared(torch.randn(2, 3, 32, 32, device=DEVICE))
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        prepared.parameters(), lr=LR_BASE, momentum=0.9, weight_decay=1e-4
    )
    scheduler = LR_SCHEDULES[lr_schedule_name](optimizer, n_epochs)
    scaler    = torch.amp.GradScaler(device="cuda", enabled=USE_AMP)   # Fix 3

    epoch_train_acc = []
    t_start         = time.perf_counter()

    for epoch in range(n_epochs):
        if epoch == OBSERVER_FREEZE_EPOCH:
            print(f"      [epoch {epoch+1}] Freezing observers + BN ...", flush=True)
            prepared = prepared.cpu()
            freeze_observers_and_bn(prepared)
            prepared = prepared.to(DEVICE)
            print(f"      [epoch {epoch+1}] Frozen. Back on {DEVICE}", flush=True)

        prepared.train()
        correct = total = 0
        t_ep    = time.perf_counter()

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs  = inputs.to(DEVICE,  non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                outputs = prepared(inputs)
                loss    = criterion(outputs, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            correct += (outputs.detach().argmax(1) == targets).sum().item()
            total   += targets.size(0)

            if (batch_idx + 1) % 100 == 0:
                elapsed = time.perf_counter() - t_ep
                eta     = elapsed / (batch_idx + 1) * (len(train_loader) - batch_idx - 1)
                print(
                    f"      [epoch {epoch+1}/{n_epochs}] "
                    f"batch {batch_idx+1}/{len(train_loader)}  "
                    f"acc={correct/total:.4f}  ETA={eta:.0f}s",
                    flush=True,
                )

        if scheduler:
            scheduler.step()
        acc = correct / total
        epoch_train_acc.append(round(acc, 6))
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        print(
            f"      [epoch {epoch+1}/{n_epochs}] DONE  acc={acc:.4f}  "
            f"time={time.perf_counter() - t_ep:.1f}s",
            flush=True,
        )

    train_time_s = time.perf_counter() - t_start
    # Fix 4: prepared is already a GraphModule — just move to CPU and eval
    trained_fakequant = prepared.cpu().eval()
    return trained_fakequant, epoch_train_acc, round(train_time_s, 2)


# ══════════════════════════════════════════════════════════════════════════════
# Model Saving
# ══════════════════════════════════════════════════════════════════════════════
def save_models(trained_fakequant: nn.Module, int8_model: nn.Module,
                obs_name: str, lr_schedule_name: str):
    os.makedirs(SAVE_DIR, exist_ok=True)
    key       = f"{obs_name}_{lr_schedule_name}"
    fq_path   = os.path.join(SAVE_DIR, f"qat_fakequant_{key}.pth")
    int8_path = os.path.join(SAVE_DIR, f"qat_int8_{key}.pth")
    torch.save(trained_fakequant.state_dict(), fq_path)
    torch.save(int8_model.state_dict(), int8_path)
    print(f"      [save] fake-quant → {fq_path}", flush=True)
    print(f"      [save] real INT8  → {int8_path}", flush=True)
    return fq_path, int8_path


# ══════════════════════════════════════════════════════════════════════════════
# Single Config Pipeline
# ══════════════════════════════════════════════════════════════════════════════
def run_one(fp32_model: nn.Module, train_loader: DataLoader,
            test_loader: DataLoader, baseline_acc: float,
            fp32_timing: dict, flops_G: float,
            obs_name: str, lr_schedule_name: str) -> dict | None:
    """
    Fix 11: clean schema — metrics only under "int8" and "fp32_reference".
    Fix 12: fp32_timing passed in (measured once outside, not re-timed here).
    Fix 7 : count_params called on the actual INT8 model.
    Fix 8 : disk_size_mb uses torch.save(model).
    """
    tag = f"{obs_name}/{lr_schedule_name}"
    print(f"\n  ┌─ [{tag}]", flush=True)

    try:
        # ── QAT training ──────────────────────────────────────────────────────
        print("      [train] QAT on GPU (fake-quant + AMP) ...", flush=True)
        trained_fq, epoch_acc, train_time_s = train_qat(
            fp32_model, train_loader, obs_name, lr_schedule_name
        )

        # ── FP32 fake-quant reference accuracy ────────────────────────────────
        print("      [eval] FP32 fake-quant reference (CPU) ...", flush=True)
        fq_metrics = evaluate(trained_fq, test_loader, device=torch.device("cpu"))
        print(f"      [eval] FP32 fake-quant acc={fq_metrics['accuracy']:.4f}", flush=True)

        # ── Convert to real INT8 ──────────────────────────────────────────────
        print("      [int8] Converting to real INT8 (onednn) ...", flush=True)
        int8_model = convert_to_int8(trained_fq)
        print("      [int8] Conversion done ✓", flush=True)

        # ── INT8 evaluation ───────────────────────────────────────────────────
        print("      [eval] Evaluating real INT8 (CPU) ...", flush=True)
        int8_metrics = evaluate(int8_model, test_loader, device=torch.device("cpu"))
        acc_drop     = baseline_acc - int8_metrics["accuracy"]
        print(
            f"      [eval] INT8 acc={int8_metrics['accuracy']:.4f}  "
            f"drop={acc_drop:+.6f}",
            flush=True,
        )

        # ── INT8 timing ───────────────────────────────────────────────────────
        print("      [time] Timing INT8 (CPU) ...", flush=True)
        int8_timing = measure_latency_cpu(int8_model, label="INT8-CPU")

        # ── Params, size ──────────────────────────────────────────────────────
        # Fix 7: count on the actual INT8 model, not fp32_model
        int8_params  = count_params(int8_model, model_label=f"int8_{tag.replace('/', '_')}")
        size_fp32_mb = disk_size_mb(fp32_model)
        size_int8_mb = disk_size_mb(int8_model)
        size_ratio   = size_fp32_mb / size_int8_mb if size_int8_mb > 0 else 0.0
        speedup      = fp32_timing["batch128_cpu_ms"] / int8_timing["batch128_cpu_ms"]

        print(
            f"      [size] FP32={size_fp32_mb:.2f} MB  INT8={size_int8_mb:.2f} MB  "
            f"ratio={size_ratio:.2f}x",
            flush=True,
        )
        print(
            f"      [speedup] INT8 vs FP32 CPU: {speedup:.2f}x  "
            f"({fp32_timing['batch128_cpu_ms']:.1f} ms → {int8_timing['batch128_cpu_ms']:.1f} ms)",
            flush=True,
        )

        # ── Save models ───────────────────────────────────────────────────────
        fq_path, int8_path = save_models(trained_fq, int8_model, obs_name, lr_schedule_name)

        print(
            f"  └─ [{tag}]  INT8 acc={int8_metrics['accuracy']:.4f}  "
            f"drop={acc_drop:+.4f}  batch={int8_timing['batch128_cpu_ms']:.1f} ms  "
            f"throughput={int8_timing['throughput_imgs_sec']:.0f} imgs/s  "
            f"speedup={speedup:.2f}x  size={size_int8_mb:.2f} MB",
            flush=True,
        )

        # ── Build canonical result entry (Fix 11: clean, no duplicates) ──────
        return {
            # ── Config metadata ───────────────────────────────────────────────
            "obs_name"             : obs_name,
            "lr_schedule"          : lr_schedule_name,
            "backend"              : "onednn-INT8-CPU",
            "seed"                 : SEED,
            "gpu"                  : GPU_NAME,
            "compute_cap"          : SM,
            "use_amp"              : USE_AMP,
            "qat_epochs"           : QAT_EPOCHS,
            "observer_freeze_epoch": OBSERVER_FREEZE_EPOCH,
            "train_time_s"         : train_time_s,
            "epoch_train_acc"      : epoch_acc,
            # ── INT8 model results ────────────────────────────────────────────
            "int8": build_entry(
                metrics      = int8_metrics,
                params       = int8_params,
                size_mb      = size_int8_mb,
                flops_G      = flops_G,
                inference_ms = int8_timing,
            ),
            # ── FP32 fake-quant reference ─────────────────────────────────────
            "fp32_reference": {
                "accuracy" : round(fq_metrics["accuracy"],  6),
                "precision": round(fq_metrics["precision"], 6),
                "recall"   : round(fq_metrics["recall"],    6),
                "f1"       : round(fq_metrics["f1"],        6),
                "inference_ms": fp32_timing,   # Fix 12: shared, not re-timed
            },
            # ── Derived comparison metrics ────────────────────────────────────
            "acc_drop"             : round(acc_drop, 6),
            "size_fp32_mb"         : round(size_fp32_mb, 6),
            "size_int8_mb"         : round(size_int8_mb, 6),
            "size_reduction_x"     : round(size_ratio, 4),
            "int8_vs_fp32_speedup" : round(speedup, 4),
            # ── Saved paths ───────────────────────────────────────────────────
            "saved_fakequant_pth"  : fq_path,
            "saved_int8_pth"       : int8_path,
        }

    except Exception as exc:
        import traceback
        print(f"  └─ FAILED: {exc}", flush=True)
        traceback.print_exc()
        return None


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    SEP = "=" * 70
    print(f"\n{SEP}")
    print("  QAT — Real INT8 (onednn) — ResNet50 / CIFAR-10")
    print(f"  GPU    : {GPU_NAME}  ({SM})")
    print(f"  INT8   : onednn (CPU genuine INT8 SIMD kernels)")
    print(f"  Epochs : {QAT_EPOCHS}  |  Batch: {BATCH_SIZE}  |  Timing: {TIMING_RUNS} runs")
    print(f"  Seed   : {SEED}  |  AMP: {USE_AMP}")
    print(SEP + "\n")

    # ── Load baseline accuracy ─────────────────────────────────────────────
    with open(BASELINE_METRICS_PATH) as f:
        baseline_acc = json.load(f)["accuracy"]
    print(f"  Baseline accuracy : {baseline_acc:.4f}\n", flush=True)

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    # ── FLOPs — computed once on FP32, reused for all INT8 models ─────────
    # Quantization does not change op count.
    print("  Computing FLOPs (identical for FP32 and INT8) ...")
    flops_info = count_flops(fp32_model)
    flops_G    = flops_info["flops_G"]
    print(f"  FLOPs : {flops_G:.4f} GFLOPs  (tool: {flops_info['tool']})\n")

    # ── FP32 reference timing — measured ONCE, shared across all configs ───
    # Fix 12: avoids re-timing fp32_model inside every run_one() call,
    # which was slow and produced slightly different numbers each time.
    print("  Measuring FP32 CPU reference timing (once) ...")
    fp32_timing = measure_latency_cpu(fp32_model, label="FP32-CPU-reference")
    fp32_params = count_params(fp32_model, model_label="fp32_baseline")
    fp32_size   = disk_size_mb(fp32_model)
    fp32_metrics = evaluate(fp32_model, test_loader, device=torch.device("cpu"))
    fp32_entry   = build_entry(
        metrics      = fp32_metrics,
        params       = fp32_params,
        size_mb      = fp32_size,
        flops_G      = flops_G,
        inference_ms = fp32_timing,
    )
    print(
        f"  FP32 ref: acc={fp32_metrics['accuracy']:.4f}  "
        f"batch={fp32_timing['batch128_cpu_ms']:.1f} ms  "
        f"size={fp32_size:.2f} MB\n"
    )

    output = {"fp32_baseline": fp32_entry}

    # ── Sweep A: Observer type (lr=constant) ──────────────────────────────
    print("  ── Sweep A: Observer type (lr=constant) " + "─" * 28, flush=True)
    best_obs, best_acc_a = None, -1.0

    for obs_name in OBSERVER_QCONFIGS:
        key    = f"{obs_name}_constant"
        result = run_one(
            fp32_model, train_loader, test_loader,
            baseline_acc, fp32_timing, flops_G,
            obs_name=obs_name, lr_schedule_name="constant",
        )
        if result:
            output[key] = result
            if result["int8"]["accuracy"] > best_acc_a:
                best_acc_a = result["int8"]["accuracy"]
                best_obs   = obs_name

    if best_obs is None:
        print("  ✗ All Sweep A configs failed.", flush=True)
        return

    print(f"\n  Best observer from Sweep A: {best_obs} (acc={best_acc_a:.4f})", flush=True)

    # ── Sweep B: LR schedule (best observer from A) ────────────────────────
    print(f"\n  ── Sweep B: LR schedule (observer={best_obs}) " + "─" * 22, flush=True)
    for lr_name in LR_SCHEDULES:
        if lr_name == "constant":
            continue
        key    = f"{best_obs}_{lr_name}"
        result = run_one(
            fp32_model, train_loader, test_loader,
            baseline_acc, fp32_timing, flops_G,
            obs_name=best_obs, lr_schedule_name=lr_name,
        )
        if result:
            output[key] = result

    # ── Best result across all configs ────────────────────────────────────
    all_configs = {k: v for k, v in output.items() if k != "fp32_baseline" and v is not None}

    if all_configs:
        # Primary: highest INT8 accuracy; secondary: smallest accuracy drop
        best_key = max(all_configs, key=lambda k: all_configs[k]["int8"]["accuracy"])
        best     = all_configs[best_key]
        best_int8 = best["int8"]

        output["_best_result"] = {
            "config"              : best_key,
            "obs_name"            : best["obs_name"],
            "lr_schedule"         : best["lr_schedule"],
            "backend"             : best["backend"],
            "accuracy"            : best_int8["accuracy"],
            "f1"                  : best_int8["f1"],
            "acc_drop"            : best["acc_drop"],
            "size_mb"             : best_int8["size_mb"],
            "size_reduction_x"    : best["size_reduction_x"],
            "batch128_cpu_ms"     : best_int8["inference_ms"]["batch128_cpu_ms"],
            "throughput_imgs_sec" : best_int8["inference_ms"]["throughput_imgs_sec"],
            "int8_vs_fp32_speedup": best["int8_vs_fp32_speedup"],
            "flops_G"             : best_int8["flops_G"],
            "params"              : best_int8["params"],
            "params_nonzero"      : best_int8["params_nonzero"],
            "selection_criterion" : "highest INT8 accuracy across all sweep configs",
            "all_configs_accuracy": {
                k: v["int8"]["accuracy"] for k, v in all_configs.items()
            },
        }

    # ── Meta ──────────────────────────────────────────────────────────────
    output["_meta"] = {
        "script"          : "QAT — Real INT8 (onednn) — ResNet50 / CIFAR-10",
        "seed"            : SEED,
        "gpu"             : GPU_NAME,
        "compute_cap"     : SM,
        "use_amp"         : USE_AMP,
        "qat_epochs"      : QAT_EPOCHS,
        "observer_freeze_epoch": OBSERVER_FREEZE_EPOCH,
        "batch_size"      : BATCH_SIZE,
        "timing_runs"     : TIMING_RUNS,
        "warmup_runs"     : WARMUP_RUNS,
        "flops_tool"      : flops_info["tool"],
        "flops_note"      : (
            "FLOPs identical for FP32 and INT8 — quantization changes "
            "storage dtype, not arithmetic op count. "
            "INT8 CPU speedup comes from onednn VNNI/SIMD packing."
        ),
        "timing_note"     : (
            "CPU-only evaluation. time.perf_counter() used throughout. "
            "FP32 reference timing measured once before sweep and shared "
            "across all configs for consistency."
        ),
        "size_note"       : (
            "Both models serialized via torch.save(model) — full model "
            "including quantization config — for fair size comparison."
        ),
        "param_note"      : (
            "Params measured on actual INT8 model using canonical two-path "
            "deduplication: id(p) for nn.Parameter, module-path key for "
            "FX callable weights."
        ),
    }

    # ── Save JSON ──────────────────────────────────────────────────────────
    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\n  ✓ Results saved → {OUTPUT_JSON}")
    print(f"  ✓ Models saved  → {SAVE_DIR}/")

    # ── Console summary ────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print("  RESULTS SUMMARY")
    print(SEP)
    print(
        f"  {'Config':<28} {'INT8 Acc':>8} {'Drop':>8} "
        f"{'Single ms':>10} {'Batch ms':>9} {'Imgs/s':>9} "
        f"{'Speedup':>8} {'Size MB':>8}"
    )
    print(f"  {'-'*92}")

    for key, val in output.items():
        if key in ("fp32_baseline", "_best_result", "_meta") or val is None:
            continue
        lat = val["int8"]["inference_ms"]
        print(
            f"  {key:<28} {val['int8']['accuracy']:>8.4f} "
            f"{val['acc_drop']:>+8.4f} "
            f"{lat['single_img_cpu_ms']:>10.3f} "
            f"{lat['batch128_cpu_ms']:>9.1f} "
            f"{lat['throughput_imgs_sec']:>9.1f} "
            f"{val['int8_vs_fp32_speedup']:>7.2f}x "
            f"{val['int8']['size_mb']:>8.2f}"
        )

    if "_best_result" in output:
        br = output["_best_result"]
        print(f"\n  ★ Best config  : {br['config']}")
        print(f"    INT8 accuracy: {br['accuracy']:.4f}  (drop: {br['acc_drop']:+.4f})")
        print(f"    Speedup      : {br['int8_vs_fp32_speedup']:.2f}x vs FP32 CPU")
        print(f"    Size         : {br['size_mb']:.2f} MB  ({br['size_reduction_x']:.2f}x smaller)")

    print(f"\n{SEP}")
    print("  Notes:")
    print("  - INT8 backend  : onednn (genuine INT8 SIMD, not simulated)")
    print("  - 'fake-quant'  : standard QAT terminology — NOT fake results")
    print("  - Accuracy      : measured on converted real INT8 model (CPU)")
    print("  - Speedup       : FP32-CPU batch128 / INT8-CPU batch128")
    print("  - QAT training  : GPU + AMP (STE gradient through FakeQuantize)")
    print(f"  - NUM_WORKERS   : {NUM_WORKERS}")
    print(SEP + "\n")


if __name__ == "__main__":
    main()

[init] PyTorch : 2.11.0+cu128
[init] Device  : cuda
[init] GPU     : NVIDIA GeForce RTX 5070 Laptop GPU  (SM120)
[init] INT8    : onednn (CPU real INT8 SIMD kernels)
[init] Seed    : 42
[init] QAT backend : FX-graph (torch.ao)
[init] AMP         : True
[init] Epochs      : 3  Batch: 128  Timing runs: 100
[init] NUM_WORKERS : 0

  QAT — Real INT8 (onednn) — ResNet50 / CIFAR-10
  GPU    : NVIDIA GeForce RTX 5070 Laptop GPU  (SM120)
  INT8   : onednn (CPU genuine INT8 SIMD kernels)
  Epochs : 3  |  Batch: 128  |  Timing: 100 runs
  Seed   : 42  |  AMP: True

  Baseline accuracy : 0.9320

[load] ../__2__baseline_resnet50_cifar10.pth ...
[load] OK
  Computing FLOPs (identical for FP32 and INT8) ...
  FLOPs : 2.6232 GFLOPs  (tool: thop)

  Measuring FP32 CPU reference timing (once) ...
      [time:FP32-CPU-reference] single=14.07 ms  batch=747.1 ms  throughput=171 imgs/s
      count_params [fp32_baseline]: total=23,520,842  nonzero=23,520,842  sparsity=0.0000
  FP32 ref: acc=0.9320  batch=74